# **TASK 4 - Improvement experiments, final test, report draftt**

##**Predicting Livestock Health Problems**
##**ARNIS Group**

*BORINES, BUCAGO, DATILES, DESIO, MAIDO*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, glob

matches = glob.glob('/content/drive/MyDrive/Colab Notebooks/livestock_datasets.csv', recursive=True)
print(matches)

DATA_PATH = matches[0]
print("Using:", DATA_PATH)

['/content/drive/MyDrive/Colab Notebooks/livestock_datasets.csv']
Using: /content/drive/MyDrive/Colab Notebooks/livestock_datasets.csv


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score

##**E1. Which hyperparameters matter most for your model, and how did you choose the search ranges?**

For regression the regularization strength C is the most important factor because it determines how much the model is punished for having large coefficients. We tried values like 0.01, 0.1 1 10 and 100 which cover strong to regularization in steps of ten. For forest the number of trees (100 and 300) the maximum depth (none, 5 and 10) and the minimum leaf size (1, 3 and 5) are the most important factors because they control how complicated the trees get and how much they overfit. All the searches used the same stratified 5-fold cross-validation on the training set and macro F1 as the score.

##**E2. What improved, by how much, and is the improvement meaningful given the variability of the scores?**

Tuning did not help either model because the default settings were already the best in the search. Logistic regression had a score of 0.971 with C = 1 (the default) compared to 0.967 for C = 10, 0.961 for C = 100, 0.945 for C = 0.1 and only 0.628 for C = 0.01, where the penalty was too high. The best random forest (300 trees, no depth limit, minimum leaf size 1) had a score of 0.947 the same as the default. Differences of about 0.004 to 0.010 are smaller than the fold-to-fold spread of 0.012 so they are not important and the data is probably so easy to separate that there wasn't much room, for improvement.

*E1 and E2*

In [ ]:
# for E1
lr_pipe = Pipeline([
    ("prep", build_preprocess(numeric_cols, categorical_cols)),
    ("model", LogisticRegression(max_iter=2000, random_state=SEED))
])
lr_grid = GridSearchCV(
    lr_pipe, {"model__C": [0.01, 0.1, 1, 10, 100]}, cv=cv, scoring="f1_macro"
).fit(X_train, y_train)
print("LR best:", lr_grid.best_params_, round(lr_grid.best_score_, 3))
print(pd.DataFrame(lr_grid.cv_results_)[["param_model__C", "mean_test_score", "std_test_score"]].round(3))

rf_pipe = Pipeline([
    ("prep", build_preprocess(numeric_cols, categorical_cols)),
    ("model", RandomForestClassifier(random_state=SEED))
])
rf_grid = GridSearchCV(
    rf_pipe,
    {
        "model__n_estimators": [100, 300],
        "model__max_depth": [None, 5, 10],
        "model__min_samples_leaf": [1, 3, 5]
    },
    cv=cv,
    scoring="f1_macro"
).fit(X_train, y_train)
print("RF best:", rf_grid.best_params_, round(rf_grid.best_score_, 3))

# for E2
default_lr = cross_validate(lr_pipe, X_train, y_train, cv=cv, scoring="f1_macro")["test_score"]
print("\nDefault LR:", round(default_lr.mean(), 3), "±", round(default_lr.std(), 3))
print("Tuned LR:  ", round(lr_grid.best_score_, 3))
print("Default RF: 0.947 (from D4) | Tuned RF:", round(rf_grid.best_score_, 3))

LR best: {'model__C': 1} 0.971
   param_model__C  mean_test_score  std_test_score
0            0.01            0.628           0.015
1            0.10            0.945           0.020
2            1.00            0.971           0.012
3           10.00            0.967           0.012
4          100.00            0.961           0.005
RF best: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__n_estimators': 300} 0.947

Default LR: 0.971 ± 0.012
Tuned LR:   0.971
Default RF: 0.947 (from D4) | Tuned RF: 0.947


##**E3. Did feature engineering, feature selection or ensembling help? Show evidence, including what did not help.**

None of the three ideas gave an improvement. Adding a symptom‑count feature gave the score as the logistic regression (0.971 ± 0.012). Removing age and species (feature selection) raised the score slightly to 0.977 ± 0.013 but the 0.006 difference is smaller than the fold spread so it only shows that those two features add little and give a model. A soft‑voting ensemble of regression and random forest scored 0.966 ± 0.016 lower, than logistic regression alone so ensembling did not help.

In [ ]:
def add_count(d):
    d = d.copy()
    d["n_symptoms"] = d[["Symptom_1", "Symptom_2", "Symptom_3"]].notna().sum(axis=1)
    return d

def score(pipe):
    r = cross_validate(pipe, X_train, y_train, cv=cv, scoring="f1_macro")
    return round(r["test_score"].mean(), 3), round(r["test_score"].std(), 3)

lr = LogisticRegression(C=1, max_iter=2000, random_state=SEED)
base = Pipeline([("prep", build_preprocess(numeric_cols, categorical_cols)), ("model", lr)])

count_prep = Pipeline([("clean", FunctionTransformer(clean_temperature)),
                       ("count", FunctionTransformer(add_count)),
                       ("prep", ColumnTransformer([
                           ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                                             ("scaler", StandardScaler())]),
                            numeric_cols + ["n_symptoms"]),
                           ("cat", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="None")),
                                             ("onehot", OneHotEncoder(handle_unknown="ignore"))]),
                            categorical_cols)]))])
with_count = Pipeline([("prep", count_prep), ("model", lr)])

no_age_species = Pipeline([("prep", build_preprocess(["Body_Temperature_C"], ["Symptom_1", "Symptom_2", "Symptom_3"])),
                           ("model", lr)])

ensemble = Pipeline([("prep", build_preprocess(numeric_cols, categorical_cols)),
                     ("model", VotingClassifier([("lr", lr),
                                                 ("rf", RandomForestClassifier(n_estimators=300, random_state=SEED))],
                                                voting="soft"))])

for name, p in [("Logistic regression", base), ("+ symptom count", with_count),
                ("Without age and species", no_age_species), ("Ensemble LR + RF", ensemble)]:
    print(name, score(p))

Logistic regression (np.float64(0.971), np.float64(0.012))
+ symptom count (np.float64(0.971), np.float64(0.012))
Without age and species (np.float64(0.977), np.float64(0.013))
Ensemble LR + RF (np.float64(0.966), np.float64(0.016))


##**E4. What is the final test result (evaluated once), and how does it compare with your cross-validation estimate?**

The final model used in this analysis was the logistic regression model with C = 1 and all features and received a macro F1 score of 0.951, a balanced accuracy of 0.949, and an accuracy of 0.960 on the test set of 201 animals. This is in contrast with the cross-validation estimate which was 0.971 ± 0.012. The performance on the test set is close to the cross-validation estimate (within two standard deviations) and thus we can assume that our estimate is reliable and the model did not overfit on the training data. This test was performed once after selecting all model parameters through cross-validation. Since the data is synthetic and the symptoms practically diagnose diseases, we cannot judge the performance of the model on livestock.



In [ ]:
from sklearn.metrics import classification_report

final_model = Pipeline([("prep", build_preprocess(numeric_cols, categorical_cols)),
                        ("model", LogisticRegression(C=1, max_iter=2000, random_state=SEED))])
final_model.fit(X_train, y_train)

test_pred = final_model.predict(X_test)      # test set used ONCE
print("Test macro F1:", round(f1_score(y_test, test_pred, average="macro"), 3))
print("Test balanced accuracy:", round(balanced_accuracy_score(y_test, test_pred), 3))
print("Test accuracy:", round(accuracy_score(y_test, test_pred), 3))
print(classification_report(y_test, test_pred, digits=3))

Test macro F1: 0.951
Test balanced accuracy: 0.949
Test accuracy: 0.96
                     precision    recall  f1-score   support

            Anthrax      0.947     0.900     0.923        20
     Foot and Mouth      0.880     0.917     0.898        24
            Healthy      1.000     1.000     1.000        50
           Mastitis      0.969     0.939     0.954        33
Parasitic Infection      1.000     0.939     0.969        33
          Pneumonia      0.932     1.000     0.965        41

           accuracy                          0.960       201
          macro avg      0.955     0.949     0.951       201
       weighted avg      0.961     0.960     0.960       201



#**PUSH ON GIT**

In [28]:
TOKEN = ""
EMAIL = "juamishaniqua.desio@students.isatu.edu.ph"

!git config --global user.email "juamishaniqua.desio@students.isatu.edu.ph"
!git config --global user.name "wamichuchuu"

!rm -rf ICT110_Arnis
!git clone https://wamichuchuu:@github.com/wamichuchuu/ICT110_Arnis.git
!cp "/content/drive/MyDrive/Colab Notebooks/MLTask4.ipynb" ICT110_Arnis/

%cd ICT110_Arnis
!git add MLTask4.ipynb
!git commit -m "Add ML-Task 4 notebook"
!git push origin main
%cd /content

Cloning into 'ICT110_Arnis'...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 14 (delta 2), reused 10 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (14/14), 422.65 KiB | 8.99 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/ICT110_Arnis
[main f943278] Add ML-Task 4 notebook
 1 file changed, 1 insertion(+)
 create mode 100644 MLTask4.ipynb
remote: Invalid username or token. Password authentication is not supported for Git operations.
fatal: Authentication failed for 'https://github.com/wamichuchuu/ICT110_Arnis.git/'
/content
